In [ ]:
import pandas as pd
import numpy as np
import joblib
import os
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

df = pd.read_csv("data/processed/loans_stage1.csv", low_memory=False)
df = df.loc[:, ~df.columns.str.match(r'^Unnamed: \d+$')]
# OR We can use
df = df.loc[:, ~df.columns.str.contains('^Unnamed:')]
print(df.shape)

(1345350, 94)


In [ ]:
y = df['target']
X = df.drop(columns=['target'])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Train default rate: {y_train.mean():.2%} | Test default rate: {y_test.mean():.2%}")

Train: (1076280, 93), Test: (269070, 93)
Train default rate: 19.96% | Test default rate: 19.97%


In [3]:
def parse_term(series):
    return series.str.extract(r'(\d+)').astype(float)

def parse_emp_length(series):
    mapping = {
        '< 1 year': 0, '1 year': 1, '2 years': 2, '3 years': 3, '4 years': 4,
        '5 years': 5, '6 years': 6, '7 years': 7, '8 years': 8, '9 years': 9,
        '10+ years': 10,
    }
    return series.map(mapping)

def credit_history_months(issue_d, earliest_cr_line):
    issue = pd.to_datetime(issue_d, format='%b-%Y', errors='coerce')
    earliest = pd.to_datetime(earliest_cr_line, format='%b-%Y', errors='coerce')
    days = (issue - earliest).dt.days
    return (days / 30.44).round(1)  # average days per month


def engineer_raw_fields(X):
    X = X.copy()
    if 'term' in X.columns:
        X['term_months'] = parse_term(X['term'])
        X = X.drop(columns=['term'])
    if 'emp_length' in X.columns:
        X['emp_length_years'] = parse_emp_length(X['emp_length'])
        X = X.drop(columns=['emp_length'])
    if {'issue_d', 'earliest_cr_line'}.issubset(X.columns):
        X['credit_history_months'] = credit_history_months(X['issue_d'], X['earliest_cr_line'])
        X = X.drop(columns=['issue_d', 'earliest_cr_line'])
    return X

X_train = engineer_raw_fields(X_train)
X_test = engineer_raw_fields(X_test)
X_train[[c for c in ['term_months', 'emp_length_years', 'credit_history_months'] if c in X_train.columns]].describe()

,term_months,emp_length_years,credit_history_months
count,1.076280e+06,1.013461e+06,1.076280e+06
mean,4.178912e+01,5.963741e+00,1.950340e+02
std,1.026767e+01,3.691367e+00,9.004871e+01
min,3.600000e+01,0.000000e+00,3.600000e+01
25%,3.600000e+01,2.000000e+00,1.349000e+02
50%,3.600000e+01,6.000000e+00,1.770000e+02
75%,3.600000e+01,1.000000e+01,2.400000e+02
max,6.000000e+01,1.000000e+01,9.989000e+02


In [4]:
def add_ratio_features(X):
    X = X.copy()
    monthly_income = (X['annual_inc'] / 12).replace(0, np.nan)

    if 'installment' in X.columns:
        X['installment_to_income'] = X['installment'] / monthly_income
    if 'loan_amnt' in X.columns:
        X['loan_to_income'] = X['loan_amnt'] / X['annual_inc'].replace(0, np.nan)
    if 'revol_bal' in X.columns:
        X['revol_bal_to_income'] = X['revol_bal'] / X['annual_inc'].replace(0, np.nan)
    if {'open_acc', 'total_acc'}.issubset(X.columns):
        X['open_acc_ratio'] = X['open_acc'] / X['total_acc'].replace(0, np.nan)
    return X

X_train = add_ratio_features(X_train)
X_test = add_ratio_features(X_test)
new_ratio_cols = ['installment_to_income', 'loan_to_income', 'revol_bal_to_income', 'open_acc_ratio']
X_train[[c for c in new_ratio_cols if c in X_train.columns]].describe()

,installment_to_income,loan_to_income,revol_bal_to_income,open_acc_ratio
count,1.075998e+06,1.075998e+06,1.075998e+06,1.076280e+06
mean,1.541747e-01,4.345791e-01,3.752672e-01,5.026509e-01
std,2.617002e+01,7.880863e+01,7.061361e+01,1.771153e-01
min,6.912000e-05,1.714286e-04,0.000000e+00,0.000000e+00
25%,4.625289e-02,1.246573e-01,9.826667e-02,3.750000e-01
50%,7.221158e-02,2.000000e-01,1.793077e-01,4.827586e-01
75%,1.054110e-01,2.909091e-01,2.954000e-01,6.153846e-01
max,1.320792e+04,4.000000e+04,6.532400e+04,1.750000e+00


In [5]:
def cap_outliers(X, reference=None):
    X = X.copy()
    ref = reference if reference is not None else X

    if 'annual_inc' in X.columns:
        cap = ref['annual_inc'].quantile(0.99)
        X['annual_inc'] = X['annual_inc'].clip(upper=cap)
    if 'dti' in X.columns:
        X['dti'] = X['dti'].clip(lower=0, upper=100)
    if 'revol_util' in X.columns:
        X['revol_util'] = X['revol_util'].clip(upper=150)
    return X

X_train_capped = cap_outliers(X_train)
X_test = cap_outliers(X_test, reference=X_train)  
X_train = X_train_capped

In [6]:
GRADE_ORDER = [['A', 'B', 'C', 'D', 'E', 'F', 'G']]
SUBGRADE_ORDER = [[f'{g}{n}' for g in 'ABCDEFG' for n in range(1, 6)]]

ordinal_cols = [c for c in ['grade', 'sub_grade'] if c in X_train.columns]
onehot_cols = [c for c in [
    'home_ownership', 'verification_status', 'purpose',
    'application_type', 'initial_list_status', 'addr_state',
] if c in X_train.columns]

drop_cols = [c for c in ['zip_code'] if c in X_train.columns]
X_train = X_train.drop(columns=drop_cols)
X_test = X_test.drop(columns=drop_cols)

numeric_cols = [c for c in X_train.columns
                if c not in ordinal_cols + onehot_cols
                and pd.api.types.is_numeric_dtype(X_train[c])]

print(f"Ordinal: {ordinal_cols}")
print(f"One-hot: {onehot_cols}")
print(f"Numeric: {len(numeric_cols)} columns")

Ordinal: ['grade', 'sub_grade']
One-hot: ['home_ownership', 'verification_status', 'purpose', 'application_type', 'initial_list_status', 'addr_state']
Numeric: 84 columns


In [7]:
numeric_pipeline = Pipeline([
    ('impute', SimpleImputer(strategy='median')),
    ('scale', StandardScaler()),
])

ordinal_categories = []
if 'grade' in ordinal_cols:
    ordinal_categories.append(GRADE_ORDER[0])
if 'sub_grade' in ordinal_cols:
    ordinal_categories.append(SUBGRADE_ORDER[0])

ordinal_pipeline = Pipeline([
    ('impute', SimpleImputer(strategy='most_frequent')),
    ('encode', OrdinalEncoder(categories=ordinal_categories, handle_unknown='use_encoded_value', unknown_value=-1)),
])

onehot_pipeline = Pipeline([
    ('impute', SimpleImputer(strategy='most_frequent')),
    ('encode', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
])

preprocessor = ColumnTransformer([
    ('num', numeric_pipeline, numeric_cols),
    ('ord', ordinal_pipeline, ordinal_cols),
    ('onehot', onehot_pipeline, onehot_cols),
])

X_train_transformed = preprocessor.fit_transform(X_train)
X_test_transformed = preprocessor.transform(X_test)

print(f"Train transformed shape: {X_train_transformed.shape}")
print(f"Test transformed shape: {X_test_transformed.shape}")

Train transformed shape: (1076280, 164)
Test transformed shape: (269070, 164)


In [9]:
feature_names = preprocessor.get_feature_names_out()
X_train_df = pd.DataFrame(X_train_transformed, columns=feature_names, index=X_train.index)
X_test_df = pd.DataFrame(X_test_transformed, columns=feature_names, index=X_test.index)

variances = X_train_df.var()
near_zero_var_cols = variances[variances < 0.01].index.tolist()
print(f"Dropping {len(near_zero_var_cols)} near-zero-variance columns")

X_train_final = X_train_df.drop(columns=near_zero_var_cols)
X_test_final = X_test_df.drop(columns=near_zero_var_cols)
print(f"Final shape: {X_train_final.shape}")

Dropping 32 near-zero-variance columns
Final shape: (1076280, 132)


In [11]:
os.makedirs("src/models/artifacts", exist_ok=True)

X_train_final.assign(target=y_train.values).to_csv("data/processed/train_features.csv", index=False)
X_test_final.assign(target=y_test.values).to_csv("data/processed/test_features.csv", index=False)

joblib.dump(preprocessor, "src/models/artifacts/preprocessor.joblib")
joblib.dump(near_zero_var_cols, "src/models/artifacts/dropped_near_zero_var_cols.joblib")

['src/models/artifacts/dropped_near_zero_var_cols.joblib']